# Experimento 2 — Random Forest com Balanceamento

No primeiro experimento, foi possível perceber que o modelo apresentou uma forte tendência de favorecer a classe `Excelente`.

Apesar da acurácia relativamente alta, as métricas mostraram que o modelo teve dificuldade para identificar corretamente classes minoritárias, principalmente `Crítica`.

Esse comportamento está diretamente relacionado ao forte desbalanceamento presente no dataset.

Como a maior parte das amostras pertence à classe `Excelente`, o modelo acaba aprendendo que prever essa classe com frequência gera uma alta taxa de acerto geral, mesmo que isso prejudique a identificação das demais categorias.

Por esse motivo, o segundo experimento busca analisar o impacto do balanceamento no comportamento do modelo.

Neste experimento, serão utilizadas exatamente as mesmas quatro variáveis do Experimento 1:

- Temperature
- Orthophosphate
- Country
- Waterbody Type

Porém, agora será aplicado balanceamento utilizando:

```python
class_weight="balanced"
```

Essa abordagem faz com que o algoritmo atribua pesos maiores para classes minoritárias durante o treinamento.

Na prática, isso significa que erros cometidos em classes menos representadas passam a “custar mais” para o modelo.

O objetivo não é aumentar artificialmente a quantidade de dados, mas sim fazer com que o Random Forest dê mais atenção para classes que possuem menos amostras.

Ao final do experimento, os resultados serão comparados com o Experimento 1 para analisar:

* impacto do balanceamento;
* mudanças nas métricas;
* comportamento da matriz de confusão;
* possíveis reduções de overfitting;
* melhoria ou piora na separação das classes.

## Preparação do ambiente


In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [ ]:
!pip install imbalanced-learn

In [26]:
# preparando o ambiente para machine learning
import pandas as pd

from imblearn.over_sampling import SMOTE

from imblearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [27]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [28]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (113119, 4)
Teste: (28280, 4)


In [29]:
# pré-processamento

categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
# adição de balanceamento - class_weight
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.8380997003155969
Train Precision:
0.8792897525205349
Train Recall:
0.8380997003155969
Train F1:
0.8504456740335608
Train Confusion Matrix:
[[ 6927   228   243   162]
 [ 3524 15107   342  2705]
 [  172     4   924     6]
 [ 5252  4993   683 71847]]


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Teste com 4 variáveis - com balanceamento (class_weight)
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.651980198019802

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.22      0.44      0.30      1890
         Boa       0.29      0.23      0.25      5419
     Crítica       0.05      0.08      0.07       277
   Excelente       0.82      0.79      0.81     20694

    accuracy                           0.65     28280
   macro avg       0.35      0.38      0.36     28280
weighted avg       0.67      0.65      0.66     28280


Confusion Matrix:
[[  828   457    89   516]
 [ 1115  1227   134  2943]
 [   98    79    23    77]
 [ 1644  2510   180 16360]]


## Resultados Obtidos — Experimento 2

No segundo experimento, foi aplicado balanceamento utilizando:

```python
class_weight="balanced"
```
mantendo exatamente as mesmas quatro variáveis utilizadas anteriormente:

Temperature
Orthophosphate
Country
Waterbody Type

O principal objetivo desse experimento era analisar se o balanceamento conseguiria reduzir o favorecimento excessivo da classe `Excelente` e melhorar a identificação das classes minoritárias.

A acurácia obtida no conjunto de teste foi de aproximadamente:

``` python
0.65
```
Enquanto a acurácia de treino ficou próxima de:
``` python
0.83
```

Apesar da redução na acurácia geral em comparação ao Experimento 1, alguns comportamentos importantes começaram a aparecer.

O modelo passou a dar mais atenção para as classes minoritárias, principalmente `Atenção` e `Crítica`.

Isso pode ser observado principalmente através do aumento no recall da classe `Atenção`.

No Experimento 1, o recall da classe `Atenção` era relativamente baixo, indicando que o modelo tinha dificuldade para encontrar corretamente essas amostras.

Após o balanceamento, o modelo passou a identificar uma quantidade maior de casos pertencentes a essa classe.

Além disso, a matriz de confusão mostrou uma redução parcial da tendência extrema de empurrar praticamente tudo para `Excelente`.

Mesmo assim, ainda foi possível perceber que o modelo continua apresentando dificuldades significativas para identificar corretamente a classe `Crítica`.

O recall da classe `Crítica` aumentou levemente em relação ao experimento anterior, mas ainda permaneceu baixo.

Isso mostra que o balanceamento ajudou o modelo a olhar mais para classes minoritárias, mas não foi suficiente para resolver completamente o problema.

Outro ponto importante observado foi a queda da acurácia geral.

Isso aconteceu porque, ao equilibrar os pesos das classes, o modelo deixou de priorizar excessivamente a classe majoritária.

Como consequência, ele passou a cometer mais erros em Excelente, mas também começou a explorar mais as demais categorias.

De forma geral, o Experimento 2 mostrou que:

* o balanceamento alterou significativamente o comportamento do modelo;
* houve redução parcial do favorecimento extremo da classe Excelente;
* classes minoritárias passaram a receber mais atenção;
* o modelo ficou mais sensível a padrões menos frequentes;
* ainda existem dificuldades relevantes na separação da classe Crítica.

Esses resultados serão importantes para comparação com os próximos experimentos, principalmente aqueles que utilizarão:

* novas variáveis;
* diferentes estratégias de balanceamento;
* outros algoritmos de Machine Learning.



## Teste com smote


In [30]:
# adição de balanceamento - SMOTE
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "smote",
            SMOTE(random_state=SEED)
        ),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED
            )
        )
    ]
)

In [31]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('smote', SMOTE(random_state=42)),
                ('classifier', RandomForestClassifier(random_state=42))])

In [32]:
y_train_pred = model.predict(X_train)

In [33]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.8483543878570355
Train Precision:
0.8752343473775224
Train Recall:
0.8483543878570355
Train F1:
0.856094929965956
Train Confusion Matrix:
[[ 6704   381    61   414]
 [ 3287 14580    62  3749]
 [  227    58   759    62]
 [ 4506  4298    49 73922]]


In [34]:
y_pred = model.predict(X_test)

In [35]:
# Teste com 4 variáveis - com balanceamento (SMOTE)
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.6038896746817539

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.22      0.50      0.31      1890
         Boa       0.26      0.27      0.26      5419
     Crítica       0.05      0.18      0.08       277
   Excelente       0.84      0.71      0.77     20694

    accuracy                           0.60     28280
   macro avg       0.34      0.41      0.36     28280
weighted avg       0.68      0.60      0.63     28280


Confusion Matrix:
[[  939   404   223   324]
 [ 1315  1480   288  2336]
 [  106    75    51    45]
 [ 1900  3804   382 14608]]
